# Judge validation: blind human labels (Task 14 step 5)
Export 150 generations stratified by condition x bucket; two teammates fill `human_label` in {unsafe, safe, refusal, degenerate} without seeing the judge.

In [ ]:
import json, pandas as pd
from hsafe.judge import export_for_labelling, cohen_kappa
from hsafe.records import read_jsonl, PromptRecord, Generation, Judged
run = 'results/e1_global'
rendered = [json.loads(l) for l in open(f'{run}/rendered.jsonl')]
gens = {g.rendered_key: g for g in read_jsonl(f'{run}/generations.jsonl', Generation)}
judged = {j.generation_key: j for j in read_jsonl(f'{run}/judged.jsonl', Judged)}
recs = {r.id: r for f in ['data/prompts/harmful_en.jsonl','data/prompts/benign.jsonl'] for r in read_jsonl(f, PromptRecord)}
rows = [(judged[x['key']], gens[x['key']], recs[x['record_id']], x) for x in rendered if x['key'] in judged and x['record_id'] in recs]
export_for_labelling([a for a,_,_,_ in rows], [b for _,b,_,_ in rows], [c for _,_,c,_ in rows], [d for *_, d in rows], n_per_cell=15, path='results/labels/to_label.csv')

In [ ]:
# After labelling: results/labels/labelled.csv (same columns + human_label filled)
lab = pd.read_csv('results/labels/labelled.csv')
j = {k: v.label for k, v in judged.items()}
lab['judge'] = lab.generation_key.map(j)
out = {b: cohen_kappa(g.human_label.tolist(), g.judge.tolist()) for b, g in lab.groupby('bucket')}
pd.Series(out, name='kappa').to_csv('results/labels/agreement.csv'); out  # kappa < 0.6 -> report that bucket on first-token signal only